# Retreiver Demonstration

In [25]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGREvSS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F

In [26]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

Loading weights: 100%|██████████| 291/291 [00:01<00:00, 269.14it/s, Materializing param=model.norm.weight]                              


In [52]:
# Loading Training Data:
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841823/8841823 [01:17<00:00, 114151.07it/s]


# Data Exploration

### Query Data Exploration

In [27]:
n = 10
i = 0

print('-Query ID--------Query Text-')
for key, value in queries.items():
    print('{'+ key, ":", value, "}")
    i += 1
    if i == n:
        break

-Query ID--------Query Text-
{1185869 : )what was the immediate impact of the success of the manhattan project? }
{1185868 : _________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options: }
{597651 : what color is amber urine }
{403613 : is autoimmune hepatitis a bile acid synthesis disorder }
{1183785 : elegxo meaning }
{312651 : how much does an average person make for tutoring }
{80385 : can you use a calculator on the compass test }
{645590 : what does physical medicine do }
{645337 : what does pending mean on listing }
{186154 : feeding rice cereal how many times per day }


In [47]:
print(f"There are a total of {len(queries):,} training queries")
print(f"There are a total of {len(test_queries):,} testing queries")
print("\n")

mean_length = np.mean([len(queries[key]) for key in queries.keys()])
print(f"The queries have a mean character length of {mean_length:.2f}")
mean_length = np.mean([len(queries[key].split()) for key in queries.keys()])
print(f"The queries have a mean word count of {mean_length:.2f}")

There are a total of 502,939 training queries
There are a total of 43 testing queries


The queries have a mean character length of 33.22
The queries have a mean word count of 5.97


### Passage Data Exporation

In [28]:
n = 10
i = 0
print('-Passage ID--------Passage Text-')
for key, value in corpus.items():
    print('{' + key, ":", value['text'], "}")
    i += 1
    if i == n:
        break

-Passage ID--------Passage Text-
{0 : The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated. }
{1 : The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science. }
{2 : Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can be manmade. }
{3 : The Manhattan Project was the name for a project conducted during World War II, to develop the first atomic bomb. It refers specifically to the period of the project from 194 â¦ 2-1946 un

In [48]:
print(f"There are a total of {len(corpus):,} training passages")
print(f"There are a total of {len(test_corpus):,} testing passages")
print("\n")

mean_length = np.mean([len(corpus[key]['text']) for key in corpus.keys()])
print("The corpus (pasages) has a mean character length of {mean_length:.2f}")
mean_length = np.mean([len(corpus[key]['text'].split()) for key in corpus.keys()])
print("The corpus (pasages) has a mean word count of {mean_length:.2f}")

There are a total of 8,841,823 training passages
There are a total of 8,841,823 testing passages


The corpus (pasages) has a mean character length of {mean_length:.2f}
The corpus (pasages) has a mean word count of {mean_length:.2f}


### Qrels Data Exploration

In [32]:
n = 10
i = 0

print('-Query ID--------dict(passage ID: Score)-')
for key, value in qrels.items():
    print('{'+ key, ":", value, "}")
    i += 1
    if i == n:
        break

-Query ID--------dict(passage ID: Score)-
{1185869 : {'0': 1} }
{1185868 : {'16': 1} }
{597651 : {'49': 1} }
{403613 : {'60': 1} }
{1183785 : {'389': 1} }
{312651 : {'616': 1} }
{80385 : {'723': 1} }
{645590 : {'944': 1} }
{645337 : {'1054': 1} }
{186154 : {'1160': 1} }


In [42]:
print(len(qrels))
print(len(qrels) ==  len(queries))
print("")
print("The qrels provide the mapping of queries to relevant passage(s)")

502939
True

The qrels provide the mapping of queries to relevant passage(s)


In [43]:
# n = 200
# i = 0
# print("")
# for key, value in qrels.items():
#     if len(value) > 1:
#         print('{', key, ":", value, "}")
#     i += 1
#     if i == n:
#         break

# Retreiver Demonstration

### We begin by loading the Vector Database:

**Recall:** this vector database is constructed using the trained Passage Encoder

In [ ]:
# Loading Index
index = faiss.read_index("/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_v1.index")
print(f"Index contains {index.ntotal} vectors")

Index contains 8841823 vectors
Index has data


### Load the trained Query Encoder

In [35]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/query_encoder_v1"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ) #.to("cuda")

    emb = query_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 240.04it/s, Materializing param=pooler.dense.weight]                               


### Embedding the Quesetion

In [36]:
question = queries['8']
print(question)

# Embedding Query
q_emb = encode_query([question]).detach().cpu().numpy()
print('Embedding length: ', q_emb.shape[1])

 In humans, the normal set point for body temperature is 
Embedding length:  768


In [ ]:
K = 10
scores, ids = index.search(q_emb, K)
scores = scores[0]
ids = ids[0]

In [56]:
for i, id in enumerate(ids):
    print(scores[i])
    print(corpus[str(id)]['text'])
    print("\n\n")

0.99940026
Infants. An unexplained fever is greater cause for concern in infants and in children than in adults. Call your baby's doctor if your child is: 1  Younger than age 3 months and has a rectal temperature of 100.4 F (38 C) or higher.2  Between ages 3 to 6 months and has a temperature up to 102 F (38.9 C) and seems unusually irritable, lethargic or uncomfortable or has a temperature higher than 102 F (38.9 C).ou have a fever when your temperature rises above its normal range. What's normal for you may be a little higher or lower than the average normal temperature of 98.6 F (37 C).



0.99939084
1 Intermittent: Temperature either varies from normal to fever levels during a single day, or fever may occur one day and recur in about one to three days. 2  Remittent: Fevers come and go at regular intervals. 3  Hyperpyrexia: Fever that is equal to or above 106.7 F constitutes a medical emergency.ow-grade fevers range from about 100 F-101 F; 102 F is intermediate grade for adults but a

### It Works! 